## Publishing Messages to a Topic

### Installing Libraries and Utilities 

In [ ]:
%pip install azure-servicebus==7.14.3 openai==2.38.0 python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# loading service bus configurations
service_bus_connection_string = os.getenv("SERVICE_BUS_CONNECTION_STRING")
service_bus_topic_name = os.getenv("SERVICE_BUS_TOPIC_NAME")

# loading the azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
chat_completions_model = os.getenv("CHAT_COMPLETIONS_MODEL")

### Creating the Service Bus Client

In [6]:
from azure.servicebus import ServiceBusClient

sb_client = ServiceBusClient.from_connection_string(
    conn_str = service_bus_connection_string
)

### Creating the Request Payloads

In [7]:
questions = [
    {
        "prompt": "Tell me something about Azure Service Bus",
        "model": chat_completions_model
    },
    {
        "prompt": "Tell me something about ESG reporting",
        "model": chat_completions_model
    },
    {
        "prompt": "What are Azure Service Bus Topics and Subscriptions?",
        "model": chat_completions_model
    },
    {
        "prompt": "What are the GRI and ESRS frameworks in ESG reporting?",
        "model": chat_completions_model
    },
    {
        "prompt": "What is Microsoft Copilot Studio",
        "model": chat_completions_model
    }
]

### Sending the Payloads to the Topic

In [8]:
from azure.servicebus import ServiceBusMessage
import json
import uuid

# creating the topic sender object
topic_sender = sb_client.get_topic_sender(service_bus_topic_name)

# creating a counter variable to track the correlation_id value
i=1

for question in questions:
    if i%2==0:
        message = ServiceBusMessage(
            body = json.dumps(question),
            content_type="application/json",
            message_id=str(uuid.uuid4()),
            correlation_id=f"prompt-request-{i}",
            application_properties={
                "category": "ESG",
                "workload_type": "chat-completions"
            }
        )
    
    else:
        message = ServiceBusMessage(
            body = json.dumps(question),
            content_type="application/json",
            message_id=str(uuid.uuid4()),
            correlation_id=f"prompt-request-{i}",
            application_properties={
                "category": "Microsoft",
                "workload_type": "chat-completions"
            }
        )

    topic_sender.send_messages(message=message)
    i=i+1
